In [18]:
import pandas as pd
import numpy as np

# 1. Project Parameters (Exact uit rapport Spaansen)
n_anchors = 2
N_ed_anchor = 5.0   # kN per anker
V_ed_anchor = 8.0   # kN per anker
f_ck = 35           
gamma_c = 1.5
cracked = True

# Geometrie
s = 150             # h-op-h afstand [mm]
c = 60              # Randafstand [mm]
h_slab = 102        # Dikte element [mm]

# 2. Inladen anker data
df = pd.read_csv('ankers.csv', sep=';', index_col='AnchorID')
#anchor = 

def calculate_final(anchor, N_ed, V_ed, f_ck, s, c, n):
    
    # Geometrie factoren EN 1992-4
    hef = anchor['hef']
    s_cr = 3 * hef
    c_cr = 1.5 * hef
    A_cN0 = s_cr * s_cr
    A_cN = (s + s_cr) * (c + c_cr) # Oppervlak met 1 rand en 2 ankers
    
    # --- A. TREK: STAAL - EN 1992-4 artikel 7.2.1.3 ---
    N_rd_s = anchor['NRk_s'] / anchor['gamma_Ms_N']
    util_N_s = N_ed / N_rd_s

    # --- B. TREK: BETONKEGEL (GROEP) - EN 1992-4 artikel 7.2.1.4 ---
    k_1 = 8.9 if cracked else 12.7
    N_rk_c0 = k_1 * (f_ck**0.5) * (hef**1.5) / 1000 
    
    psi_s_N = 0.7 + 0.3 * (c / c_cr) # Rand effect
    psi_re_N = 1.0 # Geen invloed wapening
    psi_ec_N = 1.0 # Geen eccentriciteit
    
    N_rd_c = (N_rk_c0 * (A_cN / A_cN0) * psi_s_N * psi_re_N) / gamma_c
    util_N_c = (N_ed * n) / N_rd_c # Groepslast / Groepsweerstand
    
    # --- C. TREK: UITTREKKEN  - EN 1992-4 artikel 7.2.1.5 ---
    k_2 = 7.5 if cracked else 10.5
    N_rd_p = (k_2 * anchor['A_h']) * 35  / gamma_c / 1000
    util_N_p = N_ed / N_rd_p
    
    # --- D. DWARS: STAAL -  EN 1992-4 artikel 7.2.2.3 ---
    V_rd_s = anchor['VRk_s'] / anchor['gamma_Ms_V']
    util_V_s = V_ed / V_rd_s
    
    # --- E. DWARS: PRY-OUT - EN 1992-4 artikel 7.2.2.4---
    V_rd_cp = anchor['k_8'] * N_rd_c
    util_V_cp = (V_ed * n) / V_rd_cp

    # --- F. DWARS: Randbreuk - EN 1992-4 artikel 7.2.2.5---
    d_nom = anchor['d_nom']
    k_9 = 1.7 if cracked else 2.4
    #lf = +2 is addition of marking ring
    lf = anchor['length'] - anchor['lhd'] - anchor['lcrp'] +2
    alpha = 0.1 * ((lf / c)**0.5)
    beta = 0.1 * ((d_nom / c)**0.2)
    # Basiswaarde V_rk_c0
    V_rk_c0 = k_9 * (d_nom**alpha) * (lf**beta) * (f_ck**0.5) * (c**1.5) / 1000
    
    # Geometrie factoren
    A_cV0 = 4.5 * (c**2)
    # Oppervlak voor een groep van 2 ankers aan één rand:
    A_cV = (s + 3*c) * min(h_slab, 1.5*c)
    
    # c2 maximeren op 1.5 * c1
    c2 = max(c_cr, 1.5 *c)
    psi_s_V = min(1.0, 0.7 + 0.3 * c_cr / (1.5 * c))
    psi_h_V = max(1.0, (1.5 * c / h_slab)**0.5)
    psi_alpha_V = max(1.0, (1 / (0**2 + (0.5 * 1)**2))**0.5) # Kracht staat loodrecht op de rand; cos(90) en sin(90) versimpeld naar 0 en 1
    psi_re_V = 1.0 # zonder randwapening
    
    V_rd_c = (V_rk_c0 * (A_cV / A_cV0) * psi_s_V * psi_h_V * psi_alpha_V * psi_re_V) / gamma_c
    util_Vrd_C = (V_ed * n) / V_rd_c

    # --- G. INTERACTIE (Staal) ---
    util_int_steel = util_N_s**2 + util_V_s**2

    # --- H. INTERACTIE (Beton) ---
    util_N_cd = max(util_N_c, util_N_p)
    util_V_cd = max(util_V_cp, util_Vrd_C)
    util_int_concrete = (util_N_cd**1.5 + util_V_cd**1.5)

    return {
        "N_rd_s [kN]": N_rd_s,
        "N_Rd_c [kN]": N_rd_c,
        "N_Rk_0 [kN]": N_rk_c0,
        "N_rd_p [kN]": N_rd_p,

        "V_rd_s [kN]": V_rd_s,
        "V_rd_cp [kN]": V_rd_cp,
        "V_rd_c [kN]": V_rd_c,
        "V_rk_c0 [kN]": V_rk_c0,
        
        "Staalbreuk_N [%] (29.79%)": util_N_s * 100,
        "Betonkegel [%] (43.79%)": util_N_c * 100,
        "Uittrekken [%] (15.16%)": util_N_p * 100,
        "Staalbreuk_V [%] (79.45%)": util_V_s * 100,
        "Pry-out_V [%] (35.03%)": util_V_cp * 100,
        "Beton-Randbreuk_V [%] (78.50%)": util_Vrd_C * 100,
        "Interactie_Steel <=1 (0.72)": util_int_steel,
        "Interactie_Beton <=1 (0.985)": util_int_concrete
        
    }

res = calculate_final(df.loc['T-FIXX_M12x70'], N_ed_anchor, V_ed_anchor, f_ck, s, c, n_anchors)

print(f"--- VALIDATIE T-FIXX M12*70 (Waarde tussen haakjes is rapport waarde Leviat-software)---")
print(f"")
for k, v in res.items():
    print(f"{k:30}: {v:.3f}")

res = calculate_final(df.loc['CONFIXX_M12x70'], N_ed_anchor, V_ed_anchor, f_ck, s, c, n_anchors)

print(f"")
print(f"--- Resultaten Confixx M12*70 ---")
print(f"")
for k, v in res.items():
    print(f"{k:30}: {v:.3f}")

--- VALIDATIE T-FIXX M12*70 (Waarde tussen haakjes is rapport waarde Leviat-software)---

N_rd_s [kN]                   : 16.782
N_Rd_c [kN]                   : 22.835
N_Rk_0 [kN]                   : 26.016
N_rd_p [kN]                   : 32.987
V_rd_s [kN]                   : 10.069
V_rd_cp [kN]                  : 45.670
V_rd_c [kN]                   : 20.382
V_rk_c0 [kN]                  : 8.338
Staalbreuk_N [%] (29.79%)     : 29.795
Betonkegel [%] (43.79%)       : 43.792
Uittrekken [%] (15.16%)       : 15.157
Staalbreuk_V [%] (79.45%)     : 79.452
Pry-out_V [%] (35.03%)        : 35.034
Beton-Randbreuk_V [%] (78.50%): 78.502
Interactie_Steel <=1 (0.72)   : 0.720
Interactie_Beton <=1 (0.985)  : 0.985

--- Resultaten Confixx M12*70 ---

N_rd_s [kN]                   : 19.282
N_Rd_c [kN]                   : 22.901
N_Rk_0 [kN]                   : 26.266
N_rd_p [kN]                   : 33.425
V_rd_s [kN]                   : 13.897
V_rd_cp [kN]                  : 45.802
V_rd_c [kN]        